### Multiple Regression

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch


### LOad The Data

In [2]:
df = pd.read_csv('50_Startups.csv')
df.head()

,RnD,Administration,Marketing,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


### EDA 

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   RnD             50 non-null     float64
 1   Administration  50 non-null     float64
 2   Marketing       50 non-null     float64
 3   State           50 non-null     object 
 4   Profit          50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [17]:
# Convert state col to numric 
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(df['State'])
df['State']=encoder.transform(df['State'])
df.head()

,RnD,Administration,Marketing,State,Profit
0,165349.20,136897.80,471784.10,2,192261.83
1,162597.70,151377.59,443898.53,0,191792.06
2,153441.51,101145.55,407934.54,1,191050.39
3,144372.41,118671.85,383199.62,2,182901.99
4,142107.34,91391.77,366168.42,1,166187.94


### Data Prepration

In [5]:
x = df.drop('Profit',axis=1)
y = df['Profit']


In [6]:
# Split the dataset into train test
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.8,random_state=42)

In [7]:
x_train = torch.FloatTensor(x_train.values)
x_test = torch.FloatTensor(x_test.values)
y_train = torch.FloatTensor(y_train.values).unsqueeze(1)
y_test = torch.FloatTensor(y_test.values).unsqueeze(1) 

In [8]:
x_train.ndim,y_train.ndim

(2, 2)

### Build the Model

In [9]:
# Create the Model
# Sequantial is container to contain all the fully connected layers
# all the neurons from previous layer will be connected to all neurons of next layer
model = torch.nn.Sequential(
    # input layer connecting to hidden layer with 8 neurons
    torch.nn.Linear(in_features=x_train.shape[1],out_features=8),
    # SET the activation function on the first hidden layer as relu
    torch.nn.ReLU(),
    # Add Another Hidden Layer 4 neurons
    torch.nn.Linear(in_features=8,out_features=4),
    torch.nn.ReLU(),
    # Output layer  
    torch.nn.Linear(in_features=4,out_features=1)
)

In [10]:
# model information
model

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

### Define The Hyperparameters 

In [11]:
# no of epochs
epochs=10000
# loss function
loss_function=torch.nn.MSELoss()
# learning rate
lr=0.001
# Optimizer
optimizer = torch.optim.Adam(model.parameters(),lr=lr)

### Training Loop

In [12]:
#collect all The Losses
losses = []

for epoch in range(epochs):
    #enable the model training
    model.train() #This method doesnot train the Model It just enables the training mode

    #Forward Propagation
    y_pred = model(x_train)

    #cal loss using the Loss function
    loss = loss_function(y_train,y_pred)

    # zero out the old Gradients
    optimizer.zero_grad()

    #cal The Loss gradients
    loss.backward()

    #optimize the model parameters:
    optimizer.step()
    
    # Collect the losses:
    losses.append(loss)

    # Print the Training Progress
    if (epoch+1) % 500 == 0:
        print(f"epoch {epoch+1} loss : {loss}")

epoch 500 loss : 268169696.0
epoch 1000 loss : 179208384.0
epoch 1500 loss : 178324704.0
epoch 2000 loss : 177811120.0
epoch 2500 loss : 177158032.0
epoch 3000 loss : 177154864.0
epoch 3500 loss : 177151344.0
epoch 4000 loss : 177147440.0
epoch 4500 loss : 177142944.0
epoch 5000 loss : 177138032.0
epoch 5500 loss : 177132640.0
epoch 6000 loss : 177126704.0
epoch 6500 loss : 177120384.0
epoch 7000 loss : 177113648.0
epoch 7500 loss : 177106560.0
epoch 8000 loss : 177099120.0
epoch 8500 loss : 177091504.0
epoch 9000 loss : 177083616.0
epoch 9500 loss : 177075360.0
epoch 10000 loss : 177066896.0


### Evaluate the Model

In [13]:
# Enable the model Evaluation Mode
# Note : Thia Method doesnot evaluate the model it just enables the evaluation Mode
# Under the Evaluation model do not learn the anything 
model.eval()

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [14]:
# Do not cal the any Gradients
with torch.no_grad():
    
    #Predict the X_test 
    predictions = model(x_test)
    
    # Cal the Evaluation Loss
    evaluation_loss = loss_function(y_test,predictions)

    print(f"evaluation loss = {evaluation_loss}")

evaluation loss = 242118960.0


In [15]:

from sklearn.metrics import r2_score
r2 = r2_score(y_test,predictions)
r2

0.7010109424591064

In [16]:
list(model.parameters())

[Parameter containing:
 tensor([[-4.5827e-02, -2.6600e-01, -4.2229e-01,  4.4792e-01],
         [-4.8908e-02,  2.8769e-01,  8.6723e-02,  6.7230e+00],
         [-2.0625e-01,  3.0346e-01,  4.8080e-03, -4.5310e+00],
         [ 6.6643e-01,  4.7318e-01, -1.7386e-01,  9.3278e+00],
         [ 4.2682e-01,  1.8181e-01,  2.3347e-01,  5.7988e+00],
         [-3.1558e-01,  5.3668e-01,  1.3341e-01, -9.1475e+00],
         [ 2.1139e-01,  2.3841e-01,  2.5384e-01,  5.1640e+00],
         [ 6.5108e-01,  3.2137e-01,  5.2766e-01,  5.3257e+00]],
        requires_grad=True),
 Parameter containing:
 tensor([ -0.1465,   7.7452,  -5.0962,   9.4098,   6.2356, -10.0284,   7.2384,
           6.8717], requires_grad=True),
 Parameter containing:
 tensor([[ 0.1059,  0.0474, -0.1629,  0.2700,  0.1279, -0.0478, -0.0341, -0.2560],
         [ 0.0115, -0.1329,  0.2750,  0.1091,  0.1799, -0.2505,  0.1364, -0.1751],
         [-0.2138,  0.2616, -0.2384,  0.7446,  0.4939, -0.2879,  0.2231,  0.2397],
         [-0.0469, -0.2407, 

 ### Model Inference 

In [22]:
with torch.no_grad():
    print(model(torch.tensor([[165349.20,136897.80,471784.10,2]])))

tensor([[204613.7188]])


### Save The Model

In [24]:
torch.save(model.state_dict(),"Model_for_50_Startups.pth")

In [25]:
model.state_dict()

OrderedDict([('0.weight',
              tensor([[-4.5827e-02, -2.6600e-01, -4.2229e-01,  4.4792e-01],
                      [-4.8908e-02,  2.8769e-01,  8.6723e-02,  6.7230e+00],
                      [-2.0625e-01,  3.0346e-01,  4.8080e-03, -4.5310e+00],
                      [ 6.6643e-01,  4.7318e-01, -1.7386e-01,  9.3278e+00],
                      [ 4.2682e-01,  1.8181e-01,  2.3347e-01,  5.7988e+00],
                      [-3.1558e-01,  5.3668e-01,  1.3341e-01, -9.1475e+00],
                      [ 2.1139e-01,  2.3841e-01,  2.5384e-01,  5.1640e+00],
                      [ 6.5108e-01,  3.2137e-01,  5.2766e-01,  5.3257e+00]])),
             ('0.bias',
              tensor([ -0.1465,   7.7452,  -5.0962,   9.4098,   6.2356, -10.0284,   7.2384,
                        6.8717])),
             ('2.weight',
              tensor([[ 0.1059,  0.0474, -0.1629,  0.2700,  0.1279, -0.0478, -0.0341, -0.2560],
                      [ 0.0115, -0.1329,  0.2750,  0.1091,  0.1799, -0.2505,  0.1364, -0.1